In [ ]:
# Modelo Final RUL - Tier 3 Produção

Este notebook implementa o modelo final para predição de Remaining Useful Life (RUL) de motores turbofan usando a estrutura modular reorganizada.

## Objetivos
- Implementar modelo final otimizado
- Utilizar a nova estrutura modular `src/`
- Gerar avaliação completa e reprodutível
- Preparar modelo para produção

## Estrutura
1. **Setup e Imports**: Configuração do ambiente
2. **Carregamento de Dados**: Dados C-MAPSS FD001
3. **Pré-processamento**: Pipeline completo
4. **Modelo**: Arquitetura final MLP
5. **Treinamento**: Com early stopping
6. **Avaliação**: Métricas completas
7. **Resultados**: Análise e visualizações
8. **Export**: Modelo e artefatos para produção

In [ ]:
# Imports - Nova Estrutura Modular
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
import sys
import os

# Configuração do path para módulos
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root / "src"))

# Imports da estrutura modular
from src.data import (
    load_data,
    create_rul_labels,
    create_all_windows,
    split_windows_by_engine,
    DEFAULT_SENSORS,
)

from src.models import create_mlp_model, create_minmax_layer, RULMetrics

from src.utils import plot_learning_curves, plot_rul_predictions

from src.evaluation import ModelEvaluator, create_evaluation_plots

print("✅ Imports realizados com sucesso!")
print(f"📁 Projeto: {project_root}")
print(f"🔢 TensorFlow: {tf.__version__}")

In [ ]:
# Configurações Globais
RANDOM_STATE = 42
DATASET = "FD001"
MODEL_VERSION = "final-v1"

# Parâmetros otimizados (baseados em experimentos anteriores)
WINDOW_SIZE = 24
WINDOW_STRIDE = 1
RE_VALUE = 129  # Early life constant

# Hiperparâmetros do modelo
LEARNING_RATE = 0.001
L1_REG = 0.1
L2_REG = 0.2
BATCH_SIZE = 128
MAX_EPOCHS = 200
PATIENCE = 20

# Caminhos
DATA_PATH = project_root / "data" / "raw"
MODELS_PATH = project_root / "models" / "final"
RESULTS_PATH = project_root / "results"

# Criar diretórios se não existirem
MODELS_PATH.mkdir(parents=True, exist_ok=True)
(RESULTS_PATH / "figures").mkdir(parents=True, exist_ok=True)
(RESULTS_PATH / "metrics").mkdir(parents=True, exist_ok=True)

print(f"🎯 Dataset: {DATASET}")
print(f"🔧 Configurações carregadas")
print(f"📊 Window size: {WINDOW_SIZE}, stride: {WINDOW_STRIDE}")
print(f"🧠 LR: {LEARNING_RATE}, L1: {L1_REG}, L2: {L2_REG}")

In [ ]:
# 1. Carregamento e Preparação dos Dados
print("📊 Carregando dados C-MAPSS...")

# Carregar dados de treino
df_train_raw = load_data(DATA_PATH / f"train_{DATASET}.txt")
print(f"✅ Dados de treino: {df_train_raw.shape}")

# Carregar dados de teste
df_test = load_data(DATA_PATH / f"test_{DATASET}.txt")
df_rul_true = load_data(DATA_PATH / f"RUL_{DATASET}.txt", column_names=["RUL"])
df_rul_true["engine_id"] = df_rul_true.index + 1
print(f"✅ Dados de teste: {df_test.shape}")

# Criar labels RUL para treino
df_train_raw = create_rul_labels(df_train_raw, Re=RE_VALUE)

# Selecionar sensores (14 sensores conforme literatura)
features_to_keep = ["engine_id", "cycle"] + DEFAULT_SENSORS
df_train_raw = df_train_raw[features_to_keep]
df_test = df_test[features_to_keep]

print(f"📈 Sensores selecionados: {len(DEFAULT_SENSORS)}")
print(f"🔧 Features finais: {df_train_raw.shape[1] - 2}")  # -2 por engine_id e cycle

# Estatísticas básicas
print(f"\n📋 Estatísticas do dataset:")
print(f"   Engines treino: {df_train_raw['engine_id'].nunique()}")
print(f"   Engines teste: {df_test['engine_id'].nunique()}")
print(
    f"   Ciclos por engine (treino): {df_train_raw.groupby('engine_id')['cycle'].count().describe()}"
)
print(
    f"   RUL range: {df_train_raw['RUL'].min():.0f} - {df_train_raw['RUL'].max():.0f}"
)

In [ ]:
# 2. Criação de Janelas Temporais e Split Train/Val
print("🪟 Criando janelas temporais...")

# Criar todas as janelas primeiro (nova abordagem)
df_all_windows = create_all_windows(
    df_train_raw,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    sensor_cols=DEFAULT_SENSORS,
)

print(f"✅ Total de janelas criadas: {len(df_all_windows)}")
print(f"📊 Engines únicos: {df_all_windows['engine_id'].nunique()}")

# Split train/val por engine (nova função)
df_train_windows, df_val_windows = split_windows_by_engine(
    df_all_windows, test_size=0.2, random_state=RANDOM_STATE
)

# Preparar arrays para o modelo
X_train = np.array(list(df_train_windows["data_vector"]))
y_train = df_train_windows["RUL"].values

X_val = np.array(list(df_val_windows["data_vector"]))
y_val = df_val_windows["RUL"].values

print(f"\n📦 Arrays preparados:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape} | Range: {y_train.min():.0f}-{y_train.max():.0f}")
print(f"   X_val: {X_val.shape}")
print(f"   y_val: {y_val.shape} | Range: {y_val.min():.0f}-{y_val.max():.0f}")

# Verificar integridade dos dados
assert X_train.shape[0] == y_train.shape[0], "Mismatch entre X_train e y_train"
assert X_val.shape[0] == y_val.shape[0], "Mismatch entre X_val e y_val"
assert X_train.shape[1] == X_val.shape[1], "Features diferentes entre train e val"

print("✅ Verificação de integridade passou!")